<a href="https://colab.research.google.com/github/richieway/Designing-A-Synthetic-Data-Generation-For-Cognitive-Performance-Analysis/blob/main/Python_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Libraries

import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Define Parameters

np.random.seed(42)
session_levels = ["Short", "Medium", "Long"]
break_levels = ["Short", "Medium", "Long"]
difficulty_levels = ["Low", "Medium", "High"]
time_levels = ["Morning", "Afternoon", "Evening", "Night"]
sessions_per_scenario = 200

In [3]:
# Generate Dataset (21600 rows)

# 3 × 3 × 3 × 4 × 200 = 21,600 rows

data = []

for s in session_levels:
    for b in break_levels:
        for d in difficulty_levels:
            for t in time_levels:
                for i in range(sessions_per_scenario):

                    # Session Length
                    if s == "Short":
                        session_val = np.random.uniform(25, 45)
                    elif s == "Medium":
                        session_val = np.random.uniform(60, 90)
                    else:
                        session_val = np.random.uniform(120, 180)

                    # Break Duration
                    if b == "Short":
                        break_val = np.random.uniform(5, 10)
                    elif b == "Medium":
                        break_val = np.random.uniform(15, 20)
                    else:
                        break_val = np.random.uniform(25, 35)

                    # Difficulty Encoding
                    diff_val = {"Low":1, "Medium":2, "High":3}[d]

                    # Time Effect
                    time_factor = {
                        "Morning": 1.1,
                        "Afternoon": 1.0,
                        "Evening": 0.9,
                        "Night": 0.85
                    }[t]

                    # Fatigue
                    fatigue = 0.02 * session_val * diff_val

                    # Recovery
                    recovery = np.log(1 + break_val)

                    # Performance
                    performance = (100 * time_factor) - fatigue + (8 * recovery)

                    # Error Count
                    error_lambda = max(1, diff_val * session_val / 50)
                    errors = np.random.poisson(error_lambda)

                    data.append([
                        s, session_val,
                        b, break_val,
                        d,
                        t,
                        fatigue,
                        recovery,
                        performance,
                        errors
                    ])

df = pd.DataFrame(data, columns=[
    "Session Category",
    "Session Length",
    "Break Category",
    "Break Duration",
    "Difficulty",
    "Time of Day",
    "Fatigue",
    "Recovery Index",
    "Performance",
    "Error Count"
])

print(df.shape)

(21600, 10)


In [4]:
# Preprocessing

df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=[
"Session Category",
"Break Category",
"Difficulty",
"Time of Day"
], drop_first=True)

In [5]:
# Save Dataset

df.to_csv("synthetic_cognitive_dataset.csv", index=False)

In [6]:
# Descriptive Statistics

desc_stats = df.describe()
desc_stats

,Session Length,Break Duration,Fatigue,Recovery Index,Performance,Error Count
count,21600.000000,21600.000000,21600.000000,21600.000000,21600.000000,21600.000000
mean,86.768027,18.352154,3.470157,2.824442,115.375379,3.476389
std,49.219934,9.429603,2.552614,0.549643,10.861172,3.114941
min,25.000614,5.000028,0.500054,1.791764,88.743389,0.000000
25%,39.878019,8.766904,1.535811,2.279000,106.760813,1.000000
50%,75.145677,17.538508,2.710504,2.919850,114.169169,3.000000
75%,135.418092,27.574323,4.891800,3.352509,124.226933,5.000000
max,179.998558,34.999213,10.799032,3.583497,138.042690,25.000000


In [7]:
# KS TEST (Distribution Validation)

ks_results = {}

for col in ["Performance", "Fatigue", "Recovery Index", "Error Count"]:
    stat, p = stats.kstest(df[col], 'norm', args=(df[col].mean(), df[col].std()))
    ks_results[col] = {"KS Statistic": stat, "p-value": p}

ks_table = pd.DataFrame(ks_results).T
ks_table

,KS Statistic,p-value
Performance,0.057114,1.095177e-61
Fatigue,0.146433,0.000000e+00
Recovery Index,0.129206,7.680821e-315
Error Count,0.176470,0.000000e+00


In [8]:
# ANOVA (Break Duration Effect)

anova = stats.f_oneway(
df[df["Break Category"]=="Short"]["Performance"],
df[df["Break Category"]=="Medium"]["Performance"],
df[df["Break Category"]=="Long"]["Performance"]
)
anova

F_onewayResult(statistic=np.float64(1996.472662401035), pvalue=np.float64(0.0))

In [9]:
# Tukey HSD

tukey = pairwise_tukeyhsd(
    endog=df["Performance"],
    groups=df["Break Category"],
    alpha=0.05
)

print(tukey)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj  lower    upper   reject
-----------------------------------------------------
  Long Medium  -4.1374   0.0  -4.5272  -3.7476   True
  Long  Short -10.4346   0.0 -10.8244 -10.0448   True
Medium  Short  -6.2972   0.0   -6.687  -5.9074   True
-----------------------------------------------------


In [10]:
# Cohen’s d

def cohens_d(x, y):
    return (x.mean() - y.mean()) / np.sqrt((x.std()**2 + y.std()**2)/2)

d_short_long = cohens_d(
    df[df["Break Category"]=="Short"]["Performance"],
    df[df["Break Category"]=="Long"]["Performance"]
)

print("Cohen’s d:", d_short_long)

Cohen’s d: -1.044445251143843


In [11]:
# Grouped Tables

group_table = df.groupby("Break Category")["Performance"].agg(["mean","std","min","max"])
group_table

,mean,std,min,max
Break Category,,,,
Long,120.232693,9.948582,100.969220,138.042690
Medium,116.095330,9.953916,96.549591,133.767244
Short,109.798113,10.032338,88.743389,128.621220


In [12]:
# Correlation Matrix + Heatmap

corr = df.corr(numeric_only=True)
corr

,Session Length,Break Duration,Fatigue,Recovery Index,Performance,Error Count
Session Length,1.000000,0.003006,0.770032,0.003726,-0.176642,0.609630
Break Duration,0.003006,1.000000,0.000880,0.976256,0.395088,0.007287
Fatigue,0.770032,0.000880,1.000000,0.001057,-0.233117,0.799665
Recovery Index,0.003726,0.976256,0.001057,1.000000,0.404897,0.008387
Performance,-0.176642,0.395088,-0.233117,0.404897,1.000000,-0.182337
Error Count,0.609630,0.007287,0.799665,0.008387,-0.182337,1.000000


In [13]:
# Correlation Heatmap

fig = px.imshow(corr,
text_auto=True,
title="Figure 4.1: Correlation Heatmap")
fig.show()

In [14]:
# Predictive Model

X = df_model.drop("Performance", axis=1)
y = df_model["Performance"]
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Linear Regression R2:", r2_score(y_test, lr_pred))
print("Random Forest R2:", r2_score(y_test, rf_pred))

Linear Regression R2: 1.0
Random Forest R2: 0.9999250061796829


In [15]:
# Visualization

# Box Plot
fig = px.box(df, x="Break Category", y="Performance", color="Break Category")
fig.show()

# Scatter Plot
fig = px.scatter(df, x="Session Length", y="Performance", color="Difficulty")
fig.show()

# Bar Chart (Time of Day)
fig = px.bar(df.groupby("Time of Day")["Performance"].mean().reset_index().sort_values("Performance", ascending=False), x="Time of Day", y="Performance", title="Performance by Time of Day")
fig.show()